### Hyperparameter Tuning

Referenced constantly throughout this series ("tune n_estimators", "lower learning_rate") without covering how the tuning itself is actually done. Covers Grid Search, Random Search, Bayesian Optimization.

## Grid Search

#### 0. Core idea

Define a fixed grid of candidate values for each hyperparameter, exhaustively try every combination, keep whichever combination scored best on a held-out validation set (or cross-validation).

Worked example: n_estimators in {50, 100, 200}, max_depth in {3, 5, 7}. That is 3x3=9 combinations to train and evaluate, every one of them, no shortcuts.
```
(50,3) (50,5) (50,7)
(100,3) (100,5) (100,7)
(200,3) (200,5) (200,7)
```
Combinatorial explosion: add a third hyperparameter with 3 candidate values, learning_rate in {0.01, 0.1, 0.3}, and the grid jumps to 3x3x3=27 combinations. 5 hyperparameters at 3 values each is 3^5=243. Grid search's cost grows exponentially in the number of tuned hyperparameters, this is its core practical limitation.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
import numpy as np

rng = np.random.default_rng(0)
X = rng.normal(size=(100, 5))
y = (X[:, 0] + X[:, 1] > 0).astype(int)

param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [3, 5, 7],
}

grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=3)
grid.fit(X, y)

print("combinations tried:", len(grid.cv_results_["params"]))
print("best params:", grid.best_params_)
print("best score:", grid.best_score_)

## Random Search

#### 0. Core idea, and why it often beats grid search

Instead of an exhaustive grid, sample a fixed NUMBER of random combinations from the search space (a distribution per hyperparameter, not just a discrete list). Counterintuitively often more efficient than grid search, even though it looks less thorough.

Worked argument: same 2 hyperparameters, but suppose A (say learning_rate) genuinely matters a lot for performance, and B (say a minor regularization knob) barely matters at all. With grid search's 3x3=9 combinations, only 3 DISTINCT values of A ever get tested, the grid ties A and B's resolution together, wasting 6 of the 9 evaluations on redundant A-values paired with different B's.
```
grid search unique A-values tested: 3   (each tested 3 times, with different B)
random search, 9 random draws: each draw picks A independently from a continuous
  range, with high probability close to 9 distinct A-values get tested
```
Since A is the hyperparameter that actually matters, random search's 9 draws explore that important dimension roughly 3x more thoroughly than grid search's 9 evaluations did, for the identical evaluation budget. This is the actual finding from Bergstra and Bengio's original random search paper: when only a few hyperparameters matter (nearly always true in practice), random search finds equally good or better configurations than grid search using the same compute budget.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_dist = {
    "n_estimators": randint(50, 300),
    "max_depth": randint(2, 10),
}

random_search = RandomizedSearchCV(RandomForestClassifier(random_state=42), param_dist, n_iter=9, cv=3, random_state=42)
random_search.fit(X, y)

unique_depths_tried = set(p["max_depth"] for p in random_search.cv_results_["params"])
print("unique max_depth values tried:", len(unique_depths_tried), "of 9 total evaluations")
print("best params:", random_search.best_params_)
print("best score:", random_search.best_score_)

## Bayesian Optimization

#### 0. Core idea

Grid and random search both ignore what has already been learned, every evaluation is independent, picked without regard to previous results. Bayesian optimization builds a probabilistic model (commonly a Gaussian Process) of "hyperparameters -> validation score" AS IT GOES, using that model to decide where to sample next, balancing exploitation (try values near the best-so-far) against exploration (try values in still-uncertain regions of the space).

#### 1. Why this is more sample-efficient

Grid and random search waste evaluations on clearly bad regions of the space that could have been ruled out after just a few samples. Bayesian optimization's surrogate model learns the rough shape of the objective function (which regions tend to score well) after only a handful of evaluations, and concentrates the remaining budget where it is likely to pay off, this matters most when each evaluation is expensive (training a large boosted model can take minutes to hours), a regime where grid/random search's wasted evaluations are genuinely costly, not just slightly inefficient.

#### 2. Practical notes

Diminishing returns advantage when evaluations are cheap and the search space is small, the overhead of fitting the surrogate model is not worth it, random search alone is usually fine there. Most valuable specifically for expensive-to-train models (XGBoost/LightGBM/CatBoost with many trees, or any neural network) and large hyperparameter spaces. Optuna and scikit-optimize are the common libraries, `pip install optuna`, not installed in this environment.

In [ ]:
# illustrative only, optuna is not installed in this environment
# import optuna
#
# def objective(trial):
#     n_estimators = trial.suggest_int("n_estimators", 50, 300)
#     max_depth = trial.suggest_int("max_depth", 2, 10)
#     model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
#     from sklearn.model_selection import cross_val_score
#     return cross_val_score(model, X, y, cv=3).mean()
#
# study = optuna.create_study(direction="maximize")
# study.optimize(objective, n_trials=20)
# print("best params:", study.best_params)
print("see commented block above, requires: pip install optuna")